In [8]:
import pandas as pd

eventos = pd.read_csv("../data/eventos_completo.csv")
playbooks = pd.read_csv("../data/playbooks.csv")

print(eventos.head())
print(playbooks)

  indicador        data  actual  forecast  diferenca  surpresa_zscore  \
0   CPI_EUA  2023-03-14     0.4       0.4        0.0         0.000000   
1   CPI_EUA  2023-04-12     0.1       0.2       -0.1        -1.007506   
2   CPI_EUA  2023-05-10     0.4       0.4        0.0         0.000000   
3   CPI_EUA  2023-06-13     0.1       0.2       -0.1        -1.007506   
4   CPI_EUA  2023-07-12     0.2       0.3       -0.1        -1.007506   

   atencao_bruta       IAN       ICE  
0           23.0  0.200000  0.387635  
1           28.0  0.311111  0.357169  
2           22.0  0.177778  0.326825  
3           20.0  0.133333  0.287914  
4           23.0  0.200000  0.256062  
  indicador ativo_alvo direcao_se_surpresa_positiva  \
0   CPI_EUA        SPY                       vender   

  direcao_se_surpresa_negativa  
0                      comprar  


In [9]:
LIMIAR_SURPRESA = 1.0
LIMIAR_IAN = eventos["IAN"].quantile(0.75)

print(f"Novo limiar de IAN (75º percentil): {LIMIAR_IAN:.4f}")

eventos["opera"] = (eventos["surpresa_zscore"].abs() > LIMIAR_SURPRESA) & (eventos["IAN"] > LIMIAR_IAN)

print(f"Total de eventos: {len(eventos)}")
print(f"Eventos operados: {eventos['opera'].sum()}")

Novo limiar de IAN (75º percentil): 0.2667
Total de eventos: 39
Eventos operados: 5


In [10]:
def determinar_direcao(row, playbooks):
    if not row["opera"]:
        return None
    regra = playbooks[playbooks["indicador"] == row["indicador"]].iloc[0]
    if row["surpresa_zscore"] > 0:
        return regra["direcao_se_surpresa_positiva"]
    else:
        return regra["direcao_se_surpresa_negativa"]

eventos["direcao"] = eventos.apply(lambda row: determinar_direcao(row, playbooks), axis=1)

print(eventos[["data", "surpresa_zscore", "IAN", "opera", "direcao"]])

          data  surpresa_zscore       IAN  opera  direcao
0   2023-03-14         0.000000  0.200000  False      NaN
1   2023-04-12        -1.007506  0.311111   True  comprar
2   2023-05-10         0.000000  0.177778  False      NaN
3   2023-06-13        -1.007506  0.133333  False      NaN
4   2023-07-12        -1.007506  0.200000  False      NaN
5   2023-08-10         0.000000  0.200000  False      NaN
6   2023-09-13         0.000000  0.122222  False      NaN
7   2023-10-12         1.007506  0.088889  False      NaN
8   2023-11-14        -1.007506  0.144444  False      NaN
9   2023-12-12         1.007506  0.044444  False      NaN
10  2024-01-11         1.007506  0.055556  False      NaN
11  2024-02-13         1.007506  0.244444  False      NaN
12  2024-03-12         0.000000  0.144444  False      NaN
13  2024-04-10         1.007506  0.311111   True   vender
14  2024-05-15        -1.007506  0.244444  False      NaN
15  2024-06-12        -1.007506  0.111111  False      NaN
16  2024-07-11

In [11]:
eventos["tamanho_posicao"] = eventos["IAN"] * (1 + eventos["ICE"])
eventos.loc[~eventos["opera"], "tamanho_posicao"] = 0

print(eventos[["data", "IAN", "ICE", "opera", "direcao", "tamanho_posicao"]])

          data       IAN       ICE  opera  direcao  tamanho_posicao
0   2023-03-14  0.200000  0.387635  False      NaN         0.000000
1   2023-04-12  0.311111  0.357169   True  comprar         0.422230
2   2023-05-10  0.177778  0.326825  False      NaN         0.000000
3   2023-06-13  0.133333  0.287914  False      NaN         0.000000
4   2023-07-12  0.200000  0.256062  False      NaN         0.000000
5   2023-08-10  0.200000  0.223625  False      NaN         0.000000
6   2023-09-13  0.122222  0.182342  False      NaN         0.000000
7   2023-10-12  0.088889  0.148786  False      NaN         0.000000
8   2023-11-14  0.144444  0.106215  False      NaN         0.000000
9   2023-12-12  0.044444  0.071592  False      NaN         0.000000
10  2024-01-11  0.055556  0.535218  False      NaN         0.000000
11  2024-02-13  0.244444  0.767433  False      NaN         0.000000
12  2024-03-12  0.144444 -0.591468  False      NaN         0.000000
13  2024-04-10  0.311111 -0.096749   True   vend

In [12]:
operacoes = eventos[eventos["opera"]]
print(operacoes[["data", "indicador", "surpresa_zscore", "IAN", "ICE", "direcao", "tamanho_posicao"]])
print(f"\nTotal de operações: {len(operacoes)}")

          data indicador  surpresa_zscore       IAN       ICE  direcao  \
1   2023-04-12   CPI_EUA        -1.007506  0.311111  0.357169  comprar   
13  2024-04-10   CPI_EUA         1.007506  0.311111 -0.096749   vender   
33  2026-02-13   CPI_EUA        -1.007506  0.422222 -0.134588  comprar   
35  2026-04-10   CPI_EUA        -1.007506  0.333333 -0.118676  comprar   
38  2026-07-14   CPI_EUA        -3.022518  0.322222 -0.090214  comprar   

    tamanho_posicao  
1          0.422230  
13         0.281011  
33         0.365396  
35         0.293775  
38         0.293153  

Total de operações: 5


In [13]:
eventos.to_csv("../data/eventos_com_decisao.csv", index=False)